# Trabajo Práctico Integrador: Análisis de Desempeño y Gestión de Estudiantes 🎓
**Materia:** Análisis de Datos Inicial  
**Carrera:** Tecnicatura Universitaria en Programación (TUP)

**Integrantes del grupo:**
1. Cristian Krahulik
2. Tomas Mastropietro
3. Juan Segura
4. Lautaro Castillo

---

## Hito 1: Elección y Planteo 🎯

**Dataset elegido:** `Student Performance and Behaviour.csv` (Trabajando sobre `dataset_DIRTY.csv` para la fase de limpieza).

**Descripción del Dataset:**
Este conjunto de datos contiene registros detallados de estudiantes universitarios, integrando variables académicas, demográficas y socioeconómicas. Cuenta con información sobre:
*   **Factores Académicos:** GPA previo, horas de estudio semanales, tasa de asistencia, materias falladas y notas de exámenes (parcial y final).
*   **Factores de Estilo de Vida:** Horas de sueño, niveles de estrés, uso de redes sociales y participación en actividades extracurriculares.
*   **Entorno y Contexto:** Calidad de conexión a internet, calidad del espacio de estudio, nivel educativo de los padres y nivel de ingresos familiar.
*   **Factores Psicológicos:** Puntuaciones de motivación y autoeficacia.

**Objetivos del Análisis (Preguntas a responder):**
1.  **Sobre el Riesgo Académico:** ¿En qué medida el agotamiento físico y mental —representado por altos niveles de estrés y pocas horas de sueño— anula el beneficio del esfuerzo académico, prediciendo un bajo rendimiento en el examen final independientemente de cuánto tiempo dedique el alumno a estudiar?
2.  **Sobre la Brecha Digital:** ¿Cómo condiciona la calidad de la conexión a internet la efectividad de las horas de estudio semanales? ¿Existe un punto crítico en la conectividad por debajo del cual el esfuerzo del estudiante deja de verse reflejado en sus calificaciones?
3.  **Sobre Comportamientos Atípicos:** ¿Qué hábitos diferencian a los estudiantes con éxito por estudio autónomo (aquellos con baja asistencia pero notas sobresalientes) de los estudiantes con ansiedad ante los exámenes (que dedican muchísimas horas al estudio pero obtienen resultados muy bajos)?

In [1]:
# Importación de librerías necesarias
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os
import json

# Añadimos el directorio raíz al path para poder importar desde 'src'
sys.path.append(os.path.abspath(os.path.join('..')))

# Configuración visual básica para los gráficos
sns.set_theme(style="whitegrid")
pd.set_option('display.max_columns', None)

---
## Hito 2: ETL y Calidad de Datos 🧹

En esta fase realizamos un proceso avanzado de **Auditoría -> Limpieza -> Normalización -> Feature Engineering**.
Utilizamos un módulo externo `src/etl.py` para asegurar la consistencia del motor de datos.

In [2]:
from src.etl import auditar_datos

# 1. Carga del dataset con ruido
path_sucio = os.path.join('..', 'data', 'dataset_DIRTY.csv')
df_crudo = pd.read_csv(path_sucio)

# 2. Auditoría Inicial
reporte_inicial = auditar_datos(df_crudo)
print("=== REPORTE DE AUDITORÍA (DATASET SUCIO) ===")
print(json.dumps(reporte_inicial, indent=4, ensure_ascii=False))

print("\nVisualización rápida de inconsistencias (GPA negativo o Edades imposibles):")
cols_interes = ['Student_ID', 'Age', 'Previous_GPA', 'Weekly_Study_Hours']
display(df_crudo[(df_crudo['Previous_GPA'] < 0) | (df_crudo['Age'] > 100)].head(5)[cols_interes])

=== REPORTE DE AUDITORÍA (DATASET SUCIO) ===
{
    "total_filas": 500100,
    "duplicados": 100,
    "columnas_con_nulos": {
        "Previous_GPA": 10001,
        "Weekly_Study_Hours": 5002,
        "Attendance_Rate": 5001
    },
    "outliers": {
        "edad_invalida": 100,
        "gpa_invalido": 50,
        "horas_estudio_imposibles": 30
    },
    "valores_negativos_genericos": {
        "Previous_GPA": 50
    }
}

Visualización rápida de inconsistencias (GPA negativo o Edades imposibles):


,Student_ID,Age,Previous_GPA,Weekly_Study_Hours
1197,234739,150,2.58,2.0
13042,428636,150,2.40,19.2
18663,176891,26,-5.00,16.9
31246,456966,22,-5.00,20.8
33479,174766,150,3.58,23.7


### Aplicación de Limpieza e Ingeniería de Variables
Ejecutamos el motor de ETL que normaliza strings y crea nuevas variables diagnósticas.

In [3]:
from src.etl import limpiar_datos, crear_caracteristicas

# 3. Ejecución de la Limpieza y Normalización
df_clean = limpiar_datos(df_crudo)

# 4. Feature Engineering (Creación de nuevas variables)
df_final = crear_caracteristicas(df_clean)

# 5. Verificación Final
reporte_final = auditar_datos(df_final)
print("=== REPORTE DE AUDITORÍA (DATASET FINAL) ===")
print(json.dumps(reporte_final, indent=4, ensure_ascii=False))

print("\nNuevas variables creadas para el análisis:")
display(df_final[['Eficiencia_Estudio', 'Indice_Desgaste', 'Alerta_Riesgo']].head())

# Guardamos el resultado final
path_clean = os.path.join('..', 'data', 'dataset_CLEAN.csv')
df_final.to_csv(path_clean, index=False)
print(f"\n✅ Dataset limpio y enriquecido guardado en: {path_clean}")

=== REPORTE DE AUDITORÍA (DATASET FINAL) ===
{
    "total_filas": 500000,
    "duplicados": 0,
    "columnas_con_nulos": {},
    "outliers": {
        "edad_invalida": 0,
        "gpa_invalido": 0,
        "horas_estudio_imposibles": 0
    },
    "valores_negativos_genericos": {}
}

Nuevas variables creadas para el análisis:


,Eficiencia_Estudio,Indice_Desgaste,Alerta_Riesgo
0,3.368098,0.900000,False
1,3.925466,0.557143,False
2,3.021978,1.197368,False
3,3.380645,0.478873,True
4,2.871212,0.686275,False



✅ Dataset limpio y enriquecido guardado en: ../data/dataset_CLEAN.csv


---
## Hito 3: Análisis y Visualización 📊

Es momento de explorar los datos visualmente y responder a las preguntas planteadas en el Hito 1 utilizando el dataset limpio `df_final`.

In [ ]:
# Estadística Descriptiva General
df_final.describe()

### Análisis de Variables Clave
*(Espacio para que el compañero responsable del Hito 3 desarrolle los 4 gráficos mínimos requeridos por la consigna, incluyendo análisis de correlación y distribución).* 

In [ ]:
# Ejemplo: Histograma de notas finales
# plt.figure(figsize=(10, 6))
# sns.histplot(df_final['Final_Exam_Score'], kde=True)
# plt.show()

---
## Hito 4: Interfaz Gráfica (Dashboard) 🖥️

El diferencial técnico del proyecto. Se debe presentar un dashboard interactivo utilizando Streamlit que permita al usuario filtrar los datos en tiempo real.

### 🛠️ Especificaciones Técnicas
1.  **Interactividad:** Widgets para filtrar por comisiones, carreras o niveles de riesgo.
2.  **KPIs:** Métricas de aprobación y promedios generales actualizables.
3.  **Visualización:** Gráficos dinámicos basados en los filtros seleccionados.

*Nota: La aplicación se encuentra en el archivo `app.py` o dentro de la carpeta `src/`. Para ejecutarla use `streamlit run app.py`.*

---
## Hito 5: Informe de Gestión y Propuestas 🚀

Basados en la evidencia recolectada, completen el diagnóstico final:

### 1. Diagnóstico Académico
*¿Qué historias cuentan los datos?*

### 2. Propuestas de Mejora (Justificadas)
* **Propuesta A:** [Acción]
* **Propuesta B:** [Acción]

### 3. Conclusión Final
*Impacto esperado de las propuestas.*